In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import halfnorm

In [24]:
from simulators import NestedModelFamily
from simulators.benchmarks import DDM
from adapters import Adapter

In [25]:
from networks.transformers.encoder import Encoder
from networks.transformers.decoder import Decoder
from networks.transformers.gpt import BayesGPTv2

In [26]:
ddm_priors = {
    "v":        {"intercept": lambda: np.random.gamma(2., 1.),
                 "slope": lambda: 0.0},
    "a":        {"intercept": lambda: np.random.normal(-1, 0.3),
                 "slope": lambda: 0.0},
    "tau":      {"intercept": lambda: np.random.normal(-1.5, 0.3),
                 "slope": lambda: 0.0},
    "s_v":      {"intercept": lambda: halfnorm.rvs(loc=0.0, scale=1.0),
                 "slope": lambda: 0.0},
    "s_tau":    {"intercept": lambda: np.random.beta(1.0, 3.0),
                 "slope": lambda: 0.0}
}

In [27]:
ddm_full_priors = {
    "v": {"intercept": lambda: np.random.gamma(2., 1.),
          "slope": lambda: np.random.normal(0., 1.)},
    "a": {"intercept": lambda: np.random.normal(-1, 0.3),
          "slope": lambda: np.random.normal(0., 1.)},
    "tau": {"intercept": lambda: np.random.normal(-1.5, 0.3),
            "slope": lambda: np.random.normal(0., 1.)},
    "s_v": {"intercept": lambda: halfnorm.rvs(loc=0.0, scale=1.0),
            "slope": lambda: np.random.normal(0., 1.)},
    "s_tau": {"intercept": lambda: np.random.beta(1.0, 3.0),
              "slope": lambda: np.random.normal(0., 1.)}
}

In [28]:
model_family = NestedModelFamily(name="DDM", model=DDM(), prior_fun=ddm_full_priors)

In [29]:
samples = model_family.batch_sample(
    batch_size=10,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "s_tau"},
        fixed_intrinsics={}
    ),
    num_obs=500,
    flatten_param_outputs=True
)

In [30]:
samples.keys()

dict_keys(['model_names', 'design_configs', 'design_matrices', 'param_masks', 'param_matrices', 'sim_data', 'regressor_masks', 'discrete_masks', 'num_obs', 'num_regressors', 'max_num_regressors', 'max_num_categories'])

In [18]:
samples["param_matrices"].shape

(10, 80)

In [19]:
samples["param_matrices"]

array([[ 1.55751331, -0.85695527, -1.69185401,  0.08357344,  0.22093655,
         0.        ,  0.        ,  0.        ,  0.        ,  1.43846953,
         0.        ,  0.        ,  0.        ,  0.        , -1.58267795,
         0.        ,  0.        ,  0.        ,  0.        ,  0.66065787,
         0.98153575,  0.        ,  0.77803919, -2.31575129,  0.        ,
         0.70491698,  0.        ,  0.18859049, -1.2204452 ,  0.        ,
         0.36333666,  0.        , -1.06126835,  1.0553175 ,  0.        ,
        -0.02712081,  0.        ,  0.6686928 ,  0.        , -0.00836837,
         0.19820831,  0.        , -0.80360647,  0.        ,  0.79978318,
         0.11137113,  0.        ,  0.53497249,  0.        , -0.87952424,
         0.05091686,  0.        ,  1.15671202,  0.78672437,  0.        ,
        -1.88568436,  0.        ,  0.30148883,  0.12509139,  0.        ,
         0.87719011,  0.        ,  1.01962949, -0.75823455,  0.        ,
         0.        ,  0.        ,  0.        ,  0. 

In [31]:
samples["param_masks"].shape

(10, 80)

In [20]:
adapter = Adapter()

In [21]:
bayesgpt = BayesGPTv2(
    encoder_num_layers=8,
    decoder_num_layers=8,
    seed_dim=128,
    num_seeds=40
)

#### Hyperparams

In [22]:
grad_clip_norm = 5.
batch_size = 32
epochs = 100
steps_per_epoch = 100
learning_rate = 2e-4

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")